# Polarized Trees — Exploring the benchmark results

Third notebook of the synthetic workflow:

1. `datasetdemo.ipynb` — generate the fixed synthetic corpora;
2. `treesbenchmark.ipynb` — search the hyperparameters and save the results;
3. **this notebook** — explore the saved results: what the search found and which settings matter;
4. `pegcomparison.ipynb` — take settings from the top 20, keep everything fixed except the PEG formulation, and compare the formulations.

Nothing here changes the selected configuration. It reads what the benchmark saved (`benchmark_results/`) and answers questions about it:

1. **The search**: how many configurations, and how good are they?
2. **The top 20**: settings, every metric, and the same per corpus (A, B, C, overall).
3. **Formulations**: which PEG formulations are in the top 20?
4. **Hyperparameters**: how much does each one matter?
5. **`relative_h`**: a closer look, with a controlled experiment.
6. **Corpora**: do the same settings work on A, B and C?
7. **By k**: recovery by the true number of active dimensions of a text.

More questions can be added as sections: each one starts from the tables loaded in section 1.

> Run `datasetdemo.ipynb` and `treesbenchmark.ipynb` first, so that `benchmark_data/` and `benchmark_results/` exist.

## 1. Setup

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# Locate the benchmarks folder (it holds benchmark_config.py).
CWD = Path.cwd().resolve()

if (CWD / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / "benchmark_config.py").exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError(
        "Could not locate project root. "
        "Run this notebook from the notebooks directory "
        "or the project root."
    )

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from benchmark_config import (
    DIMS,
    RESULTS_DIR,
    SCALE,
    check_polartox,
    load_benchmark_corpora,
    load_benchmark_dataset,
)

check_polartox()    # which polartox and which library versions produce these results

from polartox import PolarizedTreesBenchmark, PolarizedTreesPipeline
from polartox.polarized_tree import PEG_VARIANTS

# Everything this notebook saves goes here, never over the benchmark notebook's results.
OUT_DIR = RESULTS_DIR / "exploration"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# What the benchmark saved.
report = json.loads((RESULTS_DIR / "benchmark_report.json").read_text(encoding="utf-8"))
summary = pd.read_csv(RESULTS_DIR / "benchmark_configuration_summary.csv")     # one row per configuration (A, B, C averaged)
by_corpus = pd.read_csv(RESULTS_DIR / "benchmark_results.csv")                 # one row per configuration and corpus
top = pd.read_csv(RESULTS_DIR / "top_configurations.csv")                      # the top 20

SETTINGS = ["theta_filter", "min_size_frac", "max_depth", "h", "relative_h", "theta_stop"]
METRICS = ["jaccard", "precision", "recall", "exact_match"]
STAT_NAMES = ["median", "std", "q1", "q3", "min", "max"]      # the distribution of a metric over the texts
CORPORA = list(by_corpus["corpus"].unique())

# Chart style: ink and grid colours, and categorical slots 1 and 2 (the markers differ as well, so colour is not the only cue).
INK, MUTED, GRID = "#0b0b0b", "#52514e", "#e6e5e1"
BLUE, ORANGE = "#2a78d6", "#eb6834"


def style_axes(ax):
    ax.set_facecolor("white")
    ax.grid(axis="y", color=GRID, linewidth=1)
    ax.set_axisbelow(True)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        ax.spines[side].set_color(MUTED)
    ax.tick_params(colors=MUTED, labelsize=9)

## 2. The search

How many configurations were evaluated, what the search space was, and how the mean Jaccard (over A, B and C) is distributed over all of them.

In [ ]:
search = report["benchmark"]
print(f"{search['total_configurations']} configurations evaluated ({search['strategy']} search, seed {search['seed']}), "
      f"{search['runtime_seconds']:.0f} s; selection metric: {search['selection_metric']}.")
print("\nSearch space:")
for name, values in report["search_space"].items():
    print(f"  {name}: {values}")

display(summary["jaccard"].describe().round(3).to_frame("mean Jaccard"))

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4), facecolor="white")
style_axes(ax)
ax.hist(summary["jaccard"], bins=30, color=BLUE, edgecolor="white", linewidth=1)
threshold = summary["jaccard"].sort_values(ascending=False).iloc[19]
ax.axvline(threshold, color=MUTED, linestyle="--", linewidth=1.5)
ax.text(threshold, ax.get_ylim()[1], " top 20", color=MUTED, va="top", fontsize=9)
ax.set_xlabel("mean Jaccard over A, B, C", color=MUTED)
ax.set_ylabel("configurations", color=MUTED)
ax.set_title("All evaluated configurations", loc="left", fontsize=11, color=INK)
fig.tight_layout()
fig.savefig(OUT_DIR / "search_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. The top 20

`top_configurations.csv` holds the 20 best configurations, with every metric the benchmark computed: for `jaccard`, `precision` and `recall` the mean over the texts and its distribution (median, std, quartiles q1 and q3, min, max), and the `exact_match` rate, averaged over the corpora A, B and C with equal weight. A row position here (0 to 19, rank 1 = row 0) is what `SELECTED` takes in `pegcomparison.ipynb`.

In [ ]:
# The settings and the mean of every metric.
display(top[["rank", "variant", *SETTINGS, *METRICS]].round(4))

# The distribution of jaccard, precision and recall over the texts.
for metric in ("jaccard", "precision", "recall"):
    print(f"=== {metric} ===")
    display(top[["rank", "variant", metric, *[f"{metric}_{stat}" for stat in STAT_NAMES]]].round(4))

### The top 20, per corpus

The same configurations on each corpus and overall. `benchmark_results.csv` has every configuration's metrics per corpus, `benchmark_configuration_summary.csv` the overall ones.

In [ ]:
per_corpus = pd.concat([by_corpus, summary.assign(corpus="Overall")])
per_corpus = per_corpus[per_corpus["rank"].isin(top["rank"])].assign(row=lambda d: d["rank"] - 1)

top20_by_corpus = {}
for metric in METRICS:
    print(f"=== {metric}: top 20 configurations by corpus ===")
    top20_by_corpus[metric] = per_corpus.pivot(index=["row", "variant"], columns="corpus", values=metric)[[*CORPORA, "Overall"]]
    display(top20_by_corpus[metric].rename_axis(None, axis=1).round(3))

## 4. Which PEG formulations are in the top 20?

For every formulation: how many of the top 20 use it, its best rank, and how it did over **everything that was evaluated** with it.

The search sampled only a part of the grid, so each non-PEG setting was evaluated with only some of the formulations: this table says which configurations scored best in the sample, not which formulation is best at a fixed setting. That comparison is `pegcomparison.ipynb`.

In [ ]:
formulations = summary.groupby("variant")["jaccard"].agg(
    n_configs_evaluated="count", best_jaccard="max", mean_jaccard="mean", median_jaccard="median",
)
formulations.insert(0, "best_rank", summary.groupby("variant")["rank"].min())
formulations.insert(0, "n_in_top20", summary[summary["rank"] <= 20]["variant"].value_counts())
formulations["n_in_top20"] = formulations["n_in_top20"].fillna(0).astype(int)
formulations = formulations.loc[list(PEG_VARIANTS)].reset_index().round(4)
display(formulations)

## 5. How much does each hyperparameter matter?

For every value of every hyperparameter: the number of configurations, the mean and the best mean Jaccard of the configurations that use it. Because the search is random, each value is combined with many different settings of the others, so the mean is a fair summary of how the value does on average; an interaction between two hyperparameters (see `relative_h` and `h` in the next section) is not visible here.

In [ ]:
rows = []
for parameter in [*SETTINGS, "variant"]:
    grouped = summary.groupby(parameter)["jaccard"].agg(configurations="count", mean="mean", best="max")
    for value, values in grouped.iterrows():
        rows.append({"parameter": parameter, "value": value, **values})
effects = pd.DataFrame(rows)
effects["value"] = effects["value"].astype(str)
display(effects.set_index(["parameter", "value"]).round(3))

In [ ]:
PARAMETERS = [*SETTINGS, "variant"]
fig, axes = plt.subplots(2, 4, figsize=(12, 5.6), sharey=True, facecolor="white")
for ax, parameter in zip(axes.flat, PARAMETERS):
    style_axes(ax)
    part = effects[effects["parameter"] == parameter]
    if parameter == "variant":
        part = part.set_index("value").loc[list(PEG_VARIANTS)].reset_index()
    ax.plot(part["value"], part["best"], color=ORANGE, marker="s", linewidth=2, markersize=6,
            markeredgecolor="white", markeredgewidth=1.5)
    ax.plot(part["value"], part["mean"], color=BLUE, marker="o", linewidth=2, markersize=6,
            markeredgecolor="white", markeredgewidth=1.5)
    ax.set_title(parameter, loc="left", fontsize=11, color=INK)
    ax.margins(x=0.12)
    if parameter == "variant":
        ax.tick_params(axis="x", labelrotation=45)
for ax in axes[:, 0]:
    ax.set_ylabel("mean Jaccard", color=MUTED)

legend_ax = axes.flat[-1]                   # the eighth panel carries the legend
legend_ax.axis("off")
legend_ax.plot([], [], color=BLUE, marker="o", linewidth=2, markersize=6, label="mean over the configurations")
legend_ax.plot([], [], color=ORANGE, marker="s", linewidth=2, markersize=6, label="best configuration")
legend_ax.legend(loc="center left", frameon=False, labelcolor=INK)

fig.suptitle("Mean Jaccard over A, B, C, by hyperparameter value", x=0.01, ha="left", fontsize=11, color=INK)
fig.tight_layout()
fig.savefig(OUT_DIR / "hyperparameter_effects.png", dpi=200, bbox_inches="tight")
plt.show()

## 6. `relative_h`: does it matter?

`h` is the minimum gain a split must have to be made. With `relative_h=False` the best PEG must exceed `h`; with `relative_h=True` the best PEG **divided by the node's nDFU** must exceed `h`. The two modes put `h` on different scales, so **the same `h` does not mean the same thing** and they can't be compared at a single value.

First, what the search already says: the mean Jaccard of the sampled configurations by mode and `h`.

In [ ]:
display(summary.pivot_table(index="relative_h", columns="h", values="jaccard", aggfunc="mean").round(3))
print("configurations:")
display(summary.pivot_table(index="relative_h", columns="h", values="jaccard", aggfunc="count"))

### A controlled experiment

The sample is confounded: everything else varies at random, and the best settings of one mode need not suit the other. So hold everything fixed except `relative_h` and `h`, and run the benchmark's full grid: `relative_h` × `variant` × `h` on the three corpora, with the metrics computed per corpus and averaged with equal weight.

The other settings (`theta_filter`, `min_size_frac`, `max_depth`, `theta_stop`) come from the best configuration of the search with the given `relative_h`. Those settings were chosen together with the mode of that configuration, which can favour it: run this section once with `BASE_FROM_RELATIVE_H = False` and once with `False` to see whether the conclusion depends on it.

In [ ]:
BASE_FROM_RELATIVE_H = False
H_GRID = [0.02, 0.05, 0.08, 0.10, 0.15, 0.20, 0.25, 0.30]
suffix = f"base_relative_h_{BASE_FROM_RELATIVE_H}"      # in the names of the files this section saves
N_JOBS = 4          # worker processes; each holds its own copy of the data (~150 MB)

candidates = summary[summary["relative_h"] == BASE_FROM_RELATIVE_H]
base_row = candidates.loc[candidates["jaccard"].idxmax()]
BASE = {key: base_row[key].item() for key in ["theta_filter", "min_size_frac", "max_depth", "theta_stop"]}

print(f"Other settings, from the best configuration with relative_h={BASE_FROM_RELATIVE_H} "
      f"(rank {base_row['rank']}, h={base_row['h']}, {base_row['variant']}, mean Jaccard {base_row['jaccard']:.3f}):")
print(BASE)

dataset, ground_truth, text_groups = load_benchmark_dataset()

grid = PolarizedTreesBenchmark(
    pipeline=PolarizedTreesPipeline(dims=DIMS, scale=SCALE, h=0.1, **{k: BASE[k] for k in ("theta_filter", "max_depth")}),
    annotations=dataset,
    ground_truth=ground_truth,
    text_groups=text_groups,
    search_space={
        **{key: [value] for key, value in BASE.items()},
        "variant": list(PEG_VARIANTS),
        "relative_h": [False, True],
        "h": H_GRID,
    },
    strategy="full",
    n_jobs=N_JOBS,
    verbose=False,
).run()

grid_overall = grid.get_results()            # one row per configuration: the metrics averaged over A, B, C
grid_by_corpus = grid.get_group_results()    # one row per configuration and corpus
print(f"\n{len(grid_overall)} configurations")

Mean Jaccard against `h`: one panel per PEG formulation, one line per mode, over A, B and C; the panels share the y-axis.

In [ ]:
MODES = {
    False: dict(color=BLUE, marker="o", label="relative_h = False"),
    True: dict(color=ORANGE, marker="s", label="relative_h = True"),
}
low, high = grid_overall["jaccard"].min(), grid_overall["jaccard"].max()

fig, axes = plt.subplots(2, 3, figsize=(11, 6.2), sharey=True, facecolor="white")
for ax, variant in zip(axes.flat, PEG_VARIANTS):
    style_axes(ax)
    for mode, style in MODES.items():
        rows = grid_overall[(grid_overall["variant"] == variant) & (grid_overall["relative_h"] == mode)].sort_values("h")
        ax.plot(rows["h"], rows["jaccard"], color=style["color"], marker=style["marker"],
                linewidth=2, markersize=6, markeredgecolor="white", markeredgewidth=1.5)
    ax.set_title(variant, loc="left", fontsize=11, color=INK)
    ax.set_xlabel("h", color=MUTED)
    ax.set_ylim(low - 0.03, high + 0.03)
for ax in axes[:, 0]:
    ax.set_ylabel("mean Jaccard", color=MUTED)

legend_ax = axes.flat[-1]                   # the sixth panel carries the legend
legend_ax.axis("off")
for mode, style in MODES.items():
    legend_ax.plot([], [], color=style["color"], marker=style["marker"], linewidth=2, markersize=6, label=style["label"])
legend_ax.legend(loc="center left", frameon=False, labelcolor=INK)

fig.suptitle(f"Mean Jaccard over A, B, C (other settings from the best relative_h={BASE_FROM_RELATIVE_H} configuration)",
             x=0.01, ha="left", fontsize=11, color=INK)
fig.tight_layout()
fig.savefig(OUT_DIR / f"relative_h_by_formulation_{suffix}.png", dpi=200, bbox_inches="tight")
plt.show()

Each mode at its best `h`: for every formulation and mode, the `h` with the highest overall mean Jaccard, and the Jaccard of that configuration on A, B, C and overall. The *gain* is `relative_h=True` minus `relative_h=False`, each at its own best `h`, per corpus and overall.

In [ ]:
best_h = grid_overall.loc[grid_overall.groupby(["variant", "relative_h"])["jaccard"].idxmax(), ["variant", "relative_h", "h"]]
best_h = best_h.merge(grid_by_corpus[["variant", "relative_h", "h", "corpus", "jaccard"]], on=["variant", "relative_h", "h"])
best_h = best_h.pivot(index=["variant", "relative_h", "h"], columns="corpus", values="jaccard")
best_h["Overall"] = grid_overall.set_index(["variant", "relative_h", "h"]).loc[best_h.index, "jaccard"]

best_h = best_h.reset_index().set_index("variant").loc[list(PEG_VARIANTS)].reset_index()
display(best_h.rename_axis(None, axis=1).round(3))

COLUMNS = [*CORPORA, "Overall"]
by_mode = {mode: best_h[best_h["relative_h"] == mode].set_index("variant").loc[list(PEG_VARIANTS)] for mode in (False, True)}

print("The best h of each mode:")
display(pd.DataFrame({f"h, relative_h = {mode}": by_mode[mode]["h"] for mode in by_mode}).rename_axis(None))

print("Gain of relative_h = True over relative_h = False (each at its own best h), per corpus:")
gain = (by_mode[True][COLUMNS] - by_mode[False][COLUMNS]).rename_axis(None)
display(gain.round(3))

What `h` does to the trees: leaves and depth against `h` for one formulation (`SIZE_VARIANT`), averaged over A, B and C. A larger `h` stops splitting earlier. The benchmark only reports recovery, so these come from the pipeline's diagnostics.

In [ ]:
SIZE_VARIANT = "harmonic"
corpora = load_benchmark_corpora()

rows = []
for mode in (False, True):
    for h in H_GRID:
        for corpus, (data, truth) in corpora.items():
            pipe = PolarizedTreesPipeline(dims=DIMS, scale=SCALE, variant=SIZE_VARIANT, relative_h=mode, h=h, **BASE)
            diagnostics = pipe.run_full_evaluation(data, ground_truth=truth, verbose=False)["diagnostics"]
            rows.append({"relative_h": mode, "h": h, "corpus": corpus,
                         "mean_leaves": diagnostics["mean_leaves"], "mean_depth": diagnostics["mean_depth"]})
tree_size = pd.DataFrame(rows).groupby(["relative_h", "h"], as_index=False)[["mean_leaves", "mean_depth"]].mean()

fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), facecolor="white")
for ax, column, name in zip(axes, ("mean_leaves", "mean_depth"), ("leaves per tree", "depth per tree")):
    style_axes(ax)
    for mode, style in MODES.items():
        part = tree_size[tree_size["relative_h"] == mode]
        ax.plot(part["h"], part[column], color=style["color"], marker=style["marker"], linewidth=2,
                markersize=6, markeredgecolor="white", markeredgewidth=1.5, label=style["label"])
    ax.set_title(name, loc="left", fontsize=11, color=INK)
    ax.set_xlabel("h", color=MUTED)
axes[0].legend(frameon=False, labelcolor=INK)
fig.suptitle(f"{SIZE_VARIANT}: mean over A, B, C", x=0.01, ha="left", fontsize=11, color=INK)
fig.tight_layout()
fig.savefig(OUT_DIR / f"relative_h_tree_size_{suffix}.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. Do the same settings work on A, B and C?

The corpora are three regimes: A (default), B (weak signal) and C (deep intersectional). Three questions: how hard is each, do the configurations that are good on one corpus also do well on the others, and does the best configuration differ?

In [ ]:
wide_corpora = by_corpus.pivot(index="configuration_id", columns="corpus", values="jaccard")[CORPORA]

print("Jaccard of the evaluated configurations, per corpus:")
display(wide_corpora.describe().loc[["mean", "min", "50%", "max"]].rename(index={"50%": "median"}).round(3))

print("Correlation between the corpora, over the configurations:")
corpus_correlation = wide_corpora.corr()
display(corpus_correlation.round(3))

print("The best configuration of each corpus:")
best_per_corpus = by_corpus.loc[by_corpus.groupby("corpus")["jaccard"].idxmax(), ["corpus", "rank", "variant", *SETTINGS, "jaccard"]]
display(best_per_corpus.set_index("corpus").round(3))

## 8. Recovery by the true number of active dimensions (k)

A text has between one and four truly active dimensions. For the best configuration of every formulation: mean recovery by k, over the texts of A, B and C that were analysed (a text is analysed when its nDFU reaches `theta_filter`, so the numbers of texts differ between configurations). *Precision* low means extra dimensions were added (over-splitting), *recall* low means true dimensions were missed (under-splitting).

In [ ]:
text_results = pd.read_csv(RESULTS_DIR / "benchmark_text_results.csv",
                           usecols=["configuration_id", "corpus", "text_id", "k_true", *METRICS])

best_of_formulation = summary.loc[summary.groupby("variant")["jaccard"].idxmax(), ["variant", "configuration_id"]]
texts = text_results.merge(best_of_formulation, on="configuration_id")
texts[METRICS] = texts[METRICS].astype(float)

grouped = texts.groupby(["variant", "k_true"])
recovery_by_k = grouped[METRICS].mean()
recovery_by_k.insert(0, "texts", grouped.size())

print("texts per k:")
display(recovery_by_k["texts"].unstack("k_true").loc[list(PEG_VARIANTS)])
for metric in METRICS:
    print(f"=== mean {metric} by k (the best configuration of each formulation) ===")
    display(recovery_by_k[metric].unstack("k_true").loc[list(PEG_VARIANTS)].round(3))

## 9. Save

In [ ]:
formulations.to_csv(OUT_DIR / "top20_formulations.csv", index=False)
effects.to_csv(OUT_DIR / "hyperparameter_effects.csv", index=False)
best_per_corpus.to_csv(OUT_DIR / "best_per_corpus.csv", index=False)
recovery_by_k.to_csv(OUT_DIR / "recovery_by_k.csv")
corpus_correlation.to_csv(OUT_DIR / "corpus_correlation.csv")
for metric, table in top20_by_corpus.items():
    table.to_csv(OUT_DIR / f"top20_by_corpus_{metric}.csv")

grid_overall.to_csv(OUT_DIR / f"relative_h_grid_{suffix}.csv", index=False)
grid_by_corpus.to_csv(OUT_DIR / f"relative_h_grid_by_corpus_{suffix}.csv", index=False)
best_h.to_csv(OUT_DIR / f"relative_h_best_h_{suffix}.csv", index=False)
gain.to_csv(OUT_DIR / f"relative_h_gain_by_corpus_{suffix}.csv")
tree_size.to_csv(OUT_DIR / f"relative_h_tree_size_{suffix}.csv", index=False)

print("Saved to:", OUT_DIR.resolve())